In [21]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TensorFlow for NN surrogate (F8)
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel('ERROR')  # Suppress TF warnings

DATA_DIR = Path("..") / "initial_data"

In [22]:
# ======================================
# Original Random Sampling UCB
# ======================================
# Used for F1 (flat) and F5 (solved at corner)

def suggest_ucb_point(
    X,
    y,
    beta=1.5,
    n_candidates=10_000,
    random_state=0,
    bias_point=None,
    bias_scale=0.1,
    bias_fraction=0.5,
    kernel_type="rbf",
    n_restarts=5,
):
    """
    Given current data (X, y) for ONE function, suggest the next query point x*
    using a Gaussian Process + Upper Confidence Bound (UCB) heuristic.
    
    This is the RANDOM SAMPLING approach - generates many candidates and picks best.
    Used for F1 (flat function) and F5 (solved at corner).
    """

    N, d = X.shape

    # Kernel choice (RBF vs Matérn)
    if kernel_type == "matern":
        kernel = 1.0 * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(noise_level=1e-3)
    else:
        kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
        n_restarts_optimizer=n_restarts,
    )
    gp.fit(X, y)

    # Number of candidates scales with dimension
    if d <= 4:
        n = n_candidates
    else:
        n = n_candidates * 2

    rng = np.random.default_rng(random_state)

    if bias_point is not None:
        n_biased = int(n * bias_fraction)
        n_uniform = n - n_biased

        Xcand_biased = rng.normal(loc=bias_point, scale=bias_scale, size=(n_biased, d))
        Xcand_biased = np.clip(Xcand_biased, 0, 1)

        Xcand_uniform = rng.uniform(0.0, 1.0, size=(n_uniform, d))

        Xcand = np.vstack([Xcand_biased, Xcand_uniform])
    else:
        Xcand = rng.uniform(0.0, 1.0, size=(n, d))

    mu, std = gp.predict(Xcand, return_std=True)
    ucb = mu + beta * std

    # Avoid re-using already-evaluated points
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-6),
        axis=2,
    )
    mask_any_seen = np.any(same_point, axis=1)
    ucb[mask_any_seen] = -np.inf

    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]

    return best_x, gp

In [23]:
# ======================================
# WEEK 10: GP Gradient Ascent on UCB
# ======================================
# This is the UPGRADE: instead of random sampling, we directly
# optimize the UCB acquisition function via gradient ascent.
#
# Used for F3, F4, F7 (interior optima)

def suggest_gp_gradient_point(
    X,
    y,
    beta=1.5,
    kernel_type="rbf",
    n_restarts=5,
    start_x=None,
    n_steps=100,
    lr=0.05,
    step_eps=1e-3,
):
    """
    Suggest a new point using gradient ascent on the GP's UCB acquisition.

    Instead of sampling a cloud of candidates, we:
      - start from a good point (default: best observed y so far)
      - take n_steps of gradient ascent on UCB using finite differences.

    Parameters
    ----------
    X, y : current data for ONE function
    beta : float - exploration/exploitation trade-off
    kernel_type : {"rbf", "matern"}
    n_restarts : int - GP hyperparameter optimisation restarts
    start_x : np.ndarray or None - starting point (if None, use best observed)
    n_steps : int - number of gradient-ascent iterations
    lr : float - step size
    step_eps : float - finite-difference epsilon

    Returns
    -------
    best_x : np.ndarray - point obtained after gradient ascent
    gp : GaussianProcessRegressor - fitted GP model
    """

    N, d = X.shape

    # Same kernel logic as suggest_ucb_point
    if kernel_type == "matern":
        kernel = 1.0 * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(noise_level=1e-3)
    else:
        kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=0,
        n_restarts_optimizer=n_restarts,
    )
    gp.fit(X, y)

    # Starting point
    if start_x is None:
        x = X[np.argmax(y)].astype(float).copy()
    else:
        x = np.array(start_x, dtype=float).copy()
    x = np.clip(x, 0.0, 1.0)

    for _ in range(n_steps):
        grads = np.zeros_like(x)

        # Finite-difference gradient of UCB in each dimension
        for j in range(d):
            x_plus = x.copy()
            x_minus = x.copy()

            x_plus[j] = np.clip(x_plus[j] + step_eps, 0.0, 1.0)
            x_minus[j] = np.clip(x_minus[j] - step_eps, 0.0, 1.0)

            X_pair = np.vstack([x_plus, x_minus])
            mu_pair, std_pair = gp.predict(X_pair, return_std=True)
            ucb_pair = mu_pair + beta * std_pair

            grads[j] = (ucb_pair[0] - ucb_pair[1]) / (2.0 * step_eps)

        grad_norm = np.linalg.norm(grads)
        if not np.isfinite(grad_norm) or grad_norm == 0.0:
            break

        step_vec = lr * grads / grad_norm
        x = x + step_vec
        x = np.clip(x, 0.0, 1.0)

    best_x = x
    return best_x, gp

In [24]:
# ======================================
# Neural Network Surrogate (TensorFlow)
# ======================================
# For F8 (8D) we use NN to propose starting points,
# then GP Gradient Ascent to refine.

def build_nn_surrogate(input_dim):
    """
    Build a small, regularised MLP surrogate model.
    Small network + dropout to avoid overfitting on tiny data.
    """
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(1),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss='mse',
    )
    return model


def train_nn_surrogate(X, y, epochs=500, verbose=0):
    """
    Train the NN surrogate on observed data.
    Returns the trained model and normalisation parameters.
    """
    d = X.shape[1]

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_normalized = (y - y_mean) / y_std

    model = build_nn_surrogate(d)
    model.fit(X, y_normalized, epochs=epochs, batch_size=len(X), verbose=verbose)

    return model, y_mean, y_std


def gradient_ascent_on_nn(
    model,
    start_point,
    n_steps=50,
    lr=0.005,
    clip_min=0.05,
    clip_max=0.95,
):
    """
    Run gradient ascent on the NN surrogate to find a better point.
    Uses TensorFlow's GradientTape to compute gradients.
    """
    x = tf.Variable(start_point.reshape(1, -1), dtype=tf.float32)

    for _ in range(n_steps):
        with tf.GradientTape() as tape:
            y_pred = model(x, training=False)

        grad = tape.gradient(y_pred, x)
        x.assign_add(lr * grad)
        x.assign(tf.clip_by_value(x, clip_min, clip_max))

    return x.numpy().flatten()


def suggest_nn_gp_gradient_hybrid(
    X,
    y,
    beta=1.5,
    kernel_type="rbf",
    n_restarts=5,
    n_gradient_steps=50,
    gradient_lr=0.005,
    n_steps_gp=100,
    lr_gp=0.05,
    step_eps=1e-3,
    clip_min=0.05,
    clip_max=0.95,
    random_state=0,
):
    """
    WEEK 10: NN + GP hybrid that uses GP-UCB gradient ascent.

    Steps:
      1. Train the small NN surrogate on (X, y)
      2. Use gradient_ascent_on_nn from the best observed point
      3. From that NN-refined start, run GP-UCB gradient ascent

    This "upgrades the optimiser" while keeping the same surrogates.
    """
    tf.random.set_seed(random_state)

    # 1. Train NN surrogate
    model, y_mean, y_std = train_nn_surrogate(X, y, epochs=500, verbose=0)

    # 2. Refine starting point via NN gradient ascent
    best_idx = np.argmax(y)
    start_point = X[best_idx].copy()

    nn_refined = gradient_ascent_on_nn(
        model,
        start_point,
        n_steps=n_gradient_steps,
        lr=gradient_lr,
        clip_min=clip_min,
        clip_max=clip_max,
    )

    # 3. From NN-refined point, run GP-UCB gradient ascent
    best_x, gp = suggest_gp_gradient_point(
        X,
        y,
        beta=beta,
        kernel_type=kernel_type,
        n_restarts=n_restarts,
        start_x=nn_refined,
        n_steps=n_steps_gp,
        lr=lr_gp,
        step_eps=step_eps,
    )

    source = "NN-refined start + GP-UCB gradient ascent"
    return best_x, gp, source

In [25]:
# ================================
# Load multi-round inputs/outputs
# ================================

def parse_multiline_lists(filepath):
    """
    Parse a file where each list may span multiple lines.
    """
    with open(filepath) as f:
        content = f.read()

    rounds = []
    buffer = ""
    bracket_count = 0

    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""

    return rounds


# WEEK 10: update file names
input_lines = parse_multiline_lists("inputs_m21.txt")
output_lines = parse_multiline_lists("outputs_m21.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]

n_rounds = len(inputs_rounds)
assert len(outputs_rounds) == n_rounds, "Mismatch between input/output rounds"

for r in range(n_rounds):
    assert len(inputs_rounds[r]) == 8, f"Round {r} inputs != 8"
    assert len(outputs_rounds[r]) == 8, f"Round {r} outputs != 8"

print(f"Loaded data from {n_rounds} previous week(s)")


# ============================
# WEEK 10 Configuration
# ============================
#
# STRATEGY (4 weeks left):
# - F1: GP-UCB (flat function, doesn't matter)
# - F2: GP Gradient Ascent - if no improvement, try LR corner [1,1] W11
# - F3: GP Gradient Ascent - interior optimum
# - F4: GP Gradient Ascent - interior optimum  
# - F5: MANUAL probe [1, 0.9, 1, 1] - test if d2 interior is better
# - F6: GP Gradient Ascent - if no improvement, try LR corner W11
# - F7: GP Gradient Ascent - interior optimum
# - F8: NN + GP Gradient Ascent - upgrade the optimizer
#
# Exploration phase (W10-11): Try LR corners for stuck functions
# Exploitation phase (W12-13): GP Gradient Ascent to refine

beta_map = {
    1: 1.5,    # F1 – GP-UCB on flat function
    2: 2.0,    # F2 – GP Gradient Ascent
    3: 1.0,    # F3 – GP Gradient Ascent
    4: 0.5,    # F4 – GP Gradient Ascent
    5: None,   # F5 – MANUAL probe [1, 0.9, 1, 1]
    6: 1.0,    # F6 – GP Gradient Ascent
    7: 2.0,    # F7 – GP Gradient Ascent
    8: 1.5,    # F8 – NN + GP Gradient Ascent
}

bias_fraction_map = {
    1: 0.3,    # F1 – light bias
    2: None,   # F2 – gradient ascent (no bias needed)
    3: None,   # F3 – gradient ascent (no bias needed)
    4: None,   # F4 – gradient ascent (no bias needed)
    5: None,   # F5 – MANUAL probe
    6: None,   # F6 – gradient ascent (no bias needed)
    7: None,   # F7 – gradient ascent (no bias needed)
    8: None,   # F8 – NN+GP hybrid
}

kernel_map = {
    1: "rbf",
    2: "rbf",
    3: "rbf",
    4: "rbf",
    5: "rbf",
    6: "rbf",
    7: "matern",   # F7 – Matern was part of the improvement
    8: "rbf",
}

print("\n" + "=" * 60)
print(f"WEEK {n_rounds + 1} QUERIES")
print("=" * 60)

all_queries = []
for i in range(1, 9):
    # 1. Load initial data for this function
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    X = X_init.copy()
    y = y_init.copy()

    # 2. Append every round's (x, y) for this function
    for r in range(n_rounds):
        x_prev = np.array(inputs_rounds[r][i - 1])
        y_prev = float(outputs_rounds[r][i - 1])

        print(f"Week {r + 1} {x_prev} -> {y_prev}")

        X = np.vstack([X, x_prev.reshape(1, -1)])
        y = np.append(y, y_prev)

    # 3. Best point so far
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_y_so_far = y[best_idx]

    print(f"\nFunction {i}:")

    # 4. Per-function strategy
    if i == 1:
        # --------------------------------------------------
        # F1: GP-UCB (flat function, doesn't matter)
        # --------------------------------------------------
        best_x, gp = suggest_ucb_point(
            X, y,
            beta=beta_map[i],
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=bias_fraction_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
        )
        strategy = f"GP-UCB (beta={beta_map[i]}, bias={bias_fraction_map[i]})"

    elif i in (2, 3, 4, 6, 7):
        # --------------------------------------------------
        # F2: GP Gradient Ascent
        # 9 weeks stuck, but trusting GP like other functions
        # If no improvement, try LR corner [1,1] next week
        # F3: interior optimum at ~0.4
        # F4: interior optimum at ~0.43
        # F6: Stuck since W3, but trusting GP like F3, F4, F7
        # If this doesn't work, try LR corner next week
        # F7: interior optimum
        # --------------------------------------------------
        best_x, gp = suggest_gp_gradient_point(
            X, y,
            beta=beta_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
            start_x=best_point,
            n_steps=100,
            lr=0.05,
            step_eps=1e-3,
        )
        strategy = f"GP Gradient Ascent (beta={beta_map[i]}, kernel={kernel_map[i]})"

    elif i == 5:
        # --------------------------------------------------
        # F5: MANUAL probe [1, 0.9, 1, 1]
        # Testing if d2 < 1 could give higher result than [1,1,1,1]
        # W7: [0,1,1,1] = 4440.5, W9: [1,1,1,1] = 8662.5
        # We've never tested d2 interior with d1=1
        # --------------------------------------------------
        best_x = np.array([1.0, 0.9, 1.0, 1.0])
        strategy = "MANUAL probe [1, 0.9, 1, 1] - testing d2 interior"

    elif i == 8:
        # --------------------------------------------------
        # F8: NN + GP Gradient Ascent
        # Upgrade: GP random sampling -> GP gradient ascent
        # --------------------------------------------------
        best_x, gp, source = suggest_nn_gp_gradient_hybrid(
            X, y,
            beta=beta_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
            n_gradient_steps=50,
            gradient_lr=0.005,
            n_steps_gp=100,
            lr_gp=0.05,
            step_eps=1e-3,
            clip_min=0.05,
            clip_max=0.95,
            random_state=40 + i,
        )
        strategy = f"NN + GP Gradient Ascent ({source})"

    # 5. Duplicate check + fallback to UCB (which has masking)
    is_duplicate = np.any(np.all(np.isclose(X, best_x, atol=1e-6), axis=1))
    if is_duplicate:
        print("  ⚠️ WARNING: Suggested query is a DUPLICATE! Using GP-UCB fallback (next best).")
        best_x, _ = suggest_ucb_point(
            X, y,
            beta=beta_map[i] if beta_map[i] is not None else 1.5,
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=0.5 if bias_fraction_map[i] is not None else 0.5,
            kernel_type=kernel_map[i],
            n_restarts=5,
        )
        strategy = strategy + " [UCB fallback]"

    # 6. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)
    all_queries.append(query_str)

    # 6. Report
    print(f"  Total points: {len(y)}")
    print(f"  Strategy: {strategy}")
    print(f"  Best y so far: {best_y_so_far:.6f}")
    print(f"  New query to submit: {query_str}")
    print()

# 7. Log all queries for copy-paste
print("=" * 60)
print("COPY-PASTE QUERIES (one per function)")
print("=" * 60)
for i, q in enumerate(all_queries, 1):
    print(f"F{i}: {q}")
print()
print("All queries (one per line):")
for q in all_queries:
    print(q)

Loaded data from 9 previous week(s)

WEEK 10 QUERIES
Week 1 [0.954151 0.767932] -> 9.14184425020275e-88
Week 2 [0.125971 0.826988] -> -7.249963615484764e-176
Week 3 [0.849072 0.349783] -> -4.116065399436474e-89
Week 4 [0.05 0.05] -> 7.570914060942952e-193
Week 5 [0.95 0.05] -> 4.406064392151042e-291
Week 6 [0.5 0.5] -> 2.6752879910742468e-09
Week 7 [0.05 0.95] -> 4.406064392151042e-291
Week 8 [0.95 0.95] -> 4.488845644107218e-144
Week 9 [0.70367  0.169871] -> -1.097628042677378e-100

Function 1:
  Total points: 19
  Strategy: GP-UCB (beta=1.5, bias=0.3)
  Best y so far: 0.000000
  New query to submit: 0.764825-0.025403

Week 1 [0.773956 0.438878] -> 0.33661711576593556
Week 2 [0.858598 0.697368] -> 0.5919902522467393
Week 3 [0.094177 0.975622] -> -0.05158916960088751
Week 4 [0.733108 0.822566] -> 0.5873058139167443
Week 5 [0.777682 1.      ] -> 0.18668094922203354
Week 6 [0.507533 0.796346] -> 0.528986817634964
Week 7 [0.715421 0.89494 ] -> 0.4421444149323386
Week 8 [0.703661 0.926591]